In [131]:
import requests
import json
import pandas as pd
import pprint
import os
import dotenv
from dotenv import load_dotenv
load_dotenv()


True

In [132]:
#get identifiers from DAX
ticks = ['ADS', 'BMW', 'BAS', 'DB1', 'HEI'] #, 'DPW.DE', 'DTE.DE', 'FME.DE', 'FRE.DE', 'HEI.DE', 'HEN3.DE', 'IFX.DE', 'LIN.DE', 'MRK.DE', 'MUV2.DE', 'RWE.DE', 'SAP.DE', 'SIE.DE', 'VOW3.DE']

In [133]:
#Market identifier code for Xetra Frankfurt is 'XETR'
jobs = [{'idType': 'TICKER','idValue': tick, 'micCode': 'XETR'} for tick in ticks]



In [134]:
openfigi_url = 'https://api.openfigi.com/v3/mapping'
headers = {'Content-Type': 'application/json'}
response = requests.post(openfigi_url, headers=headers, data=json.dumps(jobs))
if response.status_code != 200:
    raise Exception(f"Error: {response.status_code}, {response.text}")
result = response.json()

In [135]:
pprint.pprint(result)

[{'data': [{'compositeFIGI': 'BBG000FR1Q22',
            'exchCode': 'GY',
            'figi': 'BBG000FR1RP5',
            'marketSector': 'Equity',
            'name': 'ADIDAS AG',
            'securityDescription': 'ADS',
            'securityType': 'Common Stock',
            'securityType2': 'Common Stock',
            'shareClassFIGI': 'BBG001S8J8Q3',
            'ticker': 'ADS'}]},
 {'data': [{'compositeFIGI': 'BBG000BBX8Q0',
            'exchCode': 'GY',
            'figi': 'BBG000BBXB74',
            'marketSector': 'Equity',
            'name': 'BAYERISCHE MOTOREN WERKE AG',
            'securityDescription': 'BMW',
            'securityType': 'Common Stock',
            'securityType2': 'Common Stock',
            'shareClassFIGI': 'BBG001S676R3',
            'ticker': 'BMW'}]},
 {'data': [{'compositeFIGI': 'BBG000BBVT77',
            'exchCode': 'GY',
            'figi': 'BBG000BBVVK7',
            'marketSector': 'Equity',
            'name': 'BASF SE',
            'securit

In [136]:
#just to show the dataframe manipulation steps # type: ignore
just_dict = [d['data'][0] for d in result]
df_figi = pd.DataFrame(just_dict)
df_figi

,figi,name,ticker,exchCode,compositeFIGI,securityType,marketSector,shareClassFIGI,securityType2,securityDescription
0,BBG000FR1RP5,ADIDAS AG,ADS,GY,BBG000FR1Q22,Common Stock,Equity,BBG001S8J8Q3,Common Stock,ADS
1,BBG000BBXB74,BAYERISCHE MOTOREN WERKE AG,BMW,GY,BBG000BBX8Q0,Common Stock,Equity,BBG001S676R3,Common Stock,BMW
2,BBG000BBVVK7,BASF SE,BAS,GY,BBG000BBVT77,Common Stock,Equity,BBG001S5SYB8,Common Stock,BAS
3,BBG000D54KD0,DEUTSCHE BOERSE AG,DB1,GY,BBG000D54HT0,Common Stock,Equity,BBG001S99KC0,Common Stock,DB1
4,BBG000BC1KH6,HEIDELBERG MATERIALS AG,HEI,GY,BBG000BC1HT0,Common Stock,Equity,BBG001S68KS0,Common Stock,HEI


In [137]:
#columns of interest and set ticker as index
columns_of_interest = ['name', 'ticker', 'marketSector', 'exchCode', 'figi']
df_figi = df_figi[columns_of_interest].set_index('ticker')
df_figi

,name,marketSector,exchCode,figi
ticker,,,,
ADS,ADIDAS AG,Equity,GY,BBG000FR1RP5
BMW,BAYERISCHE MOTOREN WERKE AG,Equity,GY,BBG000BBXB74
BAS,BASF SE,Equity,GY,BBG000BBVVK7
DB1,DEUTSCHE BOERSE AG,Equity,GY,BBG000D54KD0
HEI,HEIDELBERG MATERIALS AG,Equity,GY,BBG000BC1KH6


Refinitv PermID section

In [138]:
access_token = os.getenv('API_KEY11') # get API key from environment variable

In [139]:
#API endpoint for Refinitiv PermID
url = "https://api-eit.refinitiv.com/permid/match"
headers = {
    'Content-Type': 'text/plain',
    'Accept': 'application/json',
    'X-AG-Access-Token': access_token,
    'x-openmatch-numberOfMatchesPerRecord': '1',
    'x-openmatch-dataType': 'Organization',
}

In [140]:
#the first line in the etxt field is the standard identifier. We use 'Ticker'
text_field = 'Standard Identifier\n'

for tick in ticks:
    text_field += f'TICKER:{tick}' +'&&MIC:' + 'XETR\n'
print(text_field)

Standard Identifier
TICKER:ADS&&MIC:XETR
TICKER:BMW&&MIC:XETR
TICKER:BAS&&MIC:XETR
TICKER:DB1&&MIC:XETR
TICKER:HEI&&MIC:XETR



In [141]:
response = requests.post(url, headers=headers, data=text_field)
r = response.json()

print(r)

{'ignore': '        ', 'unMatched': 0, 'matched': {'total': 5, 'excellent': 5}, 'numReceivedRecords': 5, 'numProcessedRecords': 5, 'numErrorRecords': 0, 'headersIdentifiedSuccessfully': ['standard identifier'], 'headersNotIdentified': [], 'headersSupportedWereNotSent': ['localid', 'name', 'street', 'city', 'postalcode', 'state', 'country', 'website'], 'errorCode': 0, 'errorCodeMessage': 'Success', 'resolvingTimeInMs': 87, 'requestTimeInMs': 87, 'outputContentResponse': [{'ProcessingStatus': 'OK', 'Match OpenPermID': 'https://permid.org/1-4295868725', 'Match OrgName': 'Adidas AG', 'Match Score': '100%', 'Match Level': 'Excellent', 'Match Ordinal': '1', 'Original Row Number': '2', 'Input_Standard Identifier': 'TICKER:ADS&&MIC:XETR'}, {'ProcessingStatus': 'OK', 'Match OpenPermID': 'https://permid.org/1-4295869227', 'Match OrgName': 'Bayerische Motoren Werke AG', 'Match Score': '100%', 'Match Level': 'Excellent', 'Match Ordinal': '1', 'Original Row Number': '3', 'Input_Standard Identifier'

In [142]:
pprint.pprint(r['outputContentResponse'])

[{'Input_Standard Identifier': 'TICKER:ADS&&MIC:XETR',
  'Match Level': 'Excellent',
  'Match OpenPermID': 'https://permid.org/1-4295868725',
  'Match Ordinal': '1',
  'Match OrgName': 'Adidas AG',
  'Match Score': '100%',
  'Original Row Number': '2',
  'ProcessingStatus': 'OK'},
 {'Input_Standard Identifier': 'TICKER:BMW&&MIC:XETR',
  'Match Level': 'Excellent',
  'Match OpenPermID': 'https://permid.org/1-4295869227',
  'Match Ordinal': '1',
  'Match OrgName': 'Bayerische Motoren Werke AG',
  'Match Score': '100%',
  'Original Row Number': '3',
  'ProcessingStatus': 'OK'},
 {'Input_Standard Identifier': 'TICKER:BAS&&MIC:XETR',
  'Match Level': 'Excellent',
  'Match OpenPermID': 'https://permid.org/1-4295869198',
  'Match Ordinal': '1',
  'Match OrgName': 'BASF SE',
  'Match Score': '100%',
  'Original Row Number': '4',
  'ProcessingStatus': 'OK'},
 {'Input_Standard Identifier': 'TICKER:DB1&&MIC:XETR',
  'Match Level': 'Excellent',
  'Match OpenPermID': 'https://permid.org/1-429800787

In [143]:
for company in r['outputContentResponse']:
    print(company['Match OrgName'] + ' --> ' + company['Match OpenPermID'])

Adidas AG --> https://permid.org/1-4295868725
Bayerische Motoren Werke AG --> https://permid.org/1-4295869227
BASF SE --> https://permid.org/1-4295869198
Deutsche Boerse AG --> https://permid.org/1-4298007872
Heidelberg Materials AG --> https://permid.org/1-4295868961


In [144]:
#define a function to retrieve data from the urls
def permid_data(permid_url):
    permid_headers = {
        'Accept': 'text/turtle'
    }

    permid_params = {
        'format': 'json-ld',
        'access-token': access_token
    }

    #actual request
    permid_response = requests.get(permid_url, headers=headers,params=permid_params)

    #convrt the response to JSON
    permid_data = json.loads(permid_response.content)

    return permid_data

In [145]:
#create empty dictionary
permid_dict = {}

#loop thru all tickers and put the data in dictionary

for tick, i in zip(ticks, r['outputContentResponse']):
    #the PermidID url for the ticker from the reponse earlier
    permid_url = i['Match OpenPermID']

    #'use function efined above to download data'
    data = permid_data(permid_url)

    #Put desired data in dictionary for the ticker
    permid_dict[tick] = {
        'company': data['vcard:organization-name'],
        'IPO'    : data['hasIPODate'],
        'address': data['mdaas:HeadquartersAddress'],
        'website': data['hasURL'],
        'phone'  : data['tr-org:hasHeadquartersPhoneNumber'],
        'LEI'    : data['tr-org:hasLEI'],
        'permid' : data['tr-common:hasPermId'],
        'permid_url' : permid_url
    }

In [146]:
df_permid = pd.DataFrame.from_dict(permid_dict, orient='index')

df_permid

,company,IPO,address,website,phone,LEI,permid,permid_url
ADS,Adidas AG,1997-11-28T05:00:00Z,Adi-Dassler-Strasse 1\nHERZOGENAURACH\nBAYERN\...,https://www.adidas-group.com/,499132842352,549300JSX0Z4CW0V5023,4295868725,https://permid.org/1-4295868725
BMW,Bayerische Motoren Werke AG,1926-01-01T05:00:00Z,Petuelring 130\nMUENCHEN\nBAYERN\n80809\nGerma...,https://www.bmwgroup.com/,49893820,YEH5ZCD6E441RHVHD759,4295869227,https://permid.org/1-4295869227
BAS,BASF SE,1952-01-30T05:00:00Z,Carl-Bosch-Str. 38\nLUDWIGSHAFEN AM RHEIN\nRHE...,https://www.basf.com,49621600,529900PM64WH8AF1E917,4295869198,https://permid.org/1-4295869198
DB1,Deutsche Boerse AG,2001-02-05T05:00:00Z,FRANKFURT AM MAIN\nHESSEN\n60485\nGermany\n,https://deutsche-boerse.com/,49692110,529900G3SW56SHYNPR95,4298007872,https://permid.org/1-4298007872
HEI,Heidelberg Materials AG,1997-11-28T05:00:00Z,Berliner Strasse 6\nHEIDELBERG\nBADEN-WUERTTEM...,https://www.heidelbergmaterials.com/,4962214810,LZ2C6E0W5W7LQMX5ZI37,4295868961,https://permid.org/1-4295868961


In [147]:
df_final=df_figi.join(df_permid)

df_final

,name,marketSector,exchCode,figi,company,IPO,address,website,phone,LEI,permid,permid_url
ticker,,,,,,,,,,,,
ADS,ADIDAS AG,Equity,GY,BBG000FR1RP5,Adidas AG,1997-11-28T05:00:00Z,Adi-Dassler-Strasse 1\nHERZOGENAURACH\nBAYERN\...,https://www.adidas-group.com/,499132842352,549300JSX0Z4CW0V5023,4295868725,https://permid.org/1-4295868725
BMW,BAYERISCHE MOTOREN WERKE AG,Equity,GY,BBG000BBXB74,Bayerische Motoren Werke AG,1926-01-01T05:00:00Z,Petuelring 130\nMUENCHEN\nBAYERN\n80809\nGerma...,https://www.bmwgroup.com/,49893820,YEH5ZCD6E441RHVHD759,4295869227,https://permid.org/1-4295869227
BAS,BASF SE,Equity,GY,BBG000BBVVK7,BASF SE,1952-01-30T05:00:00Z,Carl-Bosch-Str. 38\nLUDWIGSHAFEN AM RHEIN\nRHE...,https://www.basf.com,49621600,529900PM64WH8AF1E917,4295869198,https://permid.org/1-4295869198
DB1,DEUTSCHE BOERSE AG,Equity,GY,BBG000D54KD0,Deutsche Boerse AG,2001-02-05T05:00:00Z,FRANKFURT AM MAIN\nHESSEN\n60485\nGermany\n,https://deutsche-boerse.com/,49692110,529900G3SW56SHYNPR95,4298007872,https://permid.org/1-4298007872
HEI,HEIDELBERG MATERIALS AG,Equity,GY,BBG000BC1KH6,Heidelberg Materials AG,1997-11-28T05:00:00Z,Berliner Strasse 6\nHEIDELBERG\nBADEN-WUERTTEM...,https://www.heidelbergmaterials.com/,4962214810,LZ2C6E0W5W7LQMX5ZI37,4295868961,https://permid.org/1-4295868961
